# Корпоративные события MOEX

Исследование источников данных по корпоративным событиям выбранных акций.

## Дивиденды

Цель:
- проверить доступность истории дивидендов;
- изучить структуру данных;
- получить данные по выбранным акциям;
- определить поля, необходимые для хранения.

## Сплиты и консолидации

Цель:
- найти и проверить источник MOEX;
- изучить структуру данных;
- проверить наличие событий у выбранных акций;
- определить поля, необходимые для хранения.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from config.instruments import SELECTED_SECURITIES

In [ ]:
# Получение данных по дивидендам.

def get_div_data(security: str) -> pd.DataFrame:
    url = f'https://iss.moex.com/iss/securities/{security}/dividends.json'
    response = requests.get(
        url=url,
        timeout=30
    )
    if response.status_code != 200:
        response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(
        columns=data['dividends']['columns'],
        data=data['dividends']['data'],
    )
    return df

list_dfs_div: list[pd.DataFrame] = []
for selected in SELECTED_SECURITIES:
    list_dfs_div.append(get_div_data(selected))

df_div = pd.concat(list_dfs_div, ignore_index=True)
df_div.tail()

,secid,isin,registryclosedate,value,currencyid
202,AQUA,RU000A0JQTS3,2023-12-09,19,RUB
203,AQUA,RU000A0JQTS3,2024-07-08,10,RUB
204,AQUA,RU000A0JQTS3,2024-10-06,10,RUB
205,AQUA,RU000A0JQTS3,2024-12-02,20,RUB
206,AQUA,RU000A0JQTS3,2025-07-07,10,RUB


In [21]:
df_div.groupby("secid")["registryclosedate"].agg(["min", "max", "count"])

,min,max,count
secid,,,
AFKS,2014-07-17,2024-07-17,12
AQUA,2020-09-11,2025-07-07,14
GAZP,2014-07-17,2022-10-11,10
GMKN,2013-11-01,2023-12-26,21
LKOH,2013-08-15,2025-06-03,25
MOEX,2014-07-11,2025-07-10,13
NVTK,2018-10-10,2025-04-28,14
PIKK,2014-11-05,2021-05-17,5
PLZL,2017-07-17,2025-04-25,13


In [ ]:
# Справочник дроблений и консолидаций бумаг фондового рынка.

url_split = 'https://iss.moex.com/iss/statistics/engines/stock/splits.json'
response_split = requests.get(
    url=url_split,
    timeout=30,
)
response_split.raise_for_status()
data = response_split.json()

df_split = pd.DataFrame(
    columns=data['splits']['columns'],
    data=data['splits']['data'],
)

df_split = df_split[df_split['secid'].isin(SELECTED_SECURITIES)]
df_split

,tradedate,secid,before,after
39,2024-04-08,GMKN,1,100
41,2024-07-15,VTBR,5000,1
47,2025-03-27,PLZL,1,10
54,2026-04-17,T,1,10
